<center><h1> Interactive Map of Competitions </h1></center>

This notebook generates an interactive map of the WCA competitions of a desired competitor.
How to use this:
- download the WCA database in [here](https://www.worldcubeassociation.org/results/misc/export.html)
- change the WCAID down below
- run the notebook!

### WCAID

In [1]:
wcaid = '2003BRUC01' #change to desired WCAID

### Imports

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (12, 6)

In [3]:
import os
from bokeh.io import show, output_file, save, reset_output
from bokeh.models import (GMapOptions, ColumnDataSource, 
HoverTool, WMTSTileSource,CDSView, GroupFilter, Legend, Panel, Tabs, 
Button, CustomJS, DataTable, DateFormatter, TableColumn, Div)
from bokeh.plotting import gmap, figure
from bokeh.palettes import Spectral5
from bokeh.layouts import column

Results:

In [4]:
Results = pd.read_csv('WCA_export_Results.tsv', sep='\t')

Competitions:

In [5]:
Competitions = pd.read_csv('WCA_export_Competitions.tsv', sep='\t')

The <i>Results</i> and <i>Competitions</i> datasets are the main ones we are going to be using. 

For every result at each competition I will need to access the information about said competition, so I merge the two datasets:

In [6]:
df = Results.merge(Competitions, how='left', left_on='competitionId', right_on='id', validate = "m:1")
df = df.drop('id', 1)

In [7]:
df = df.rename(columns = {'name':'competitionName'})

In the early stages of the WCA, some results were not registered correctly. It can happen that final rounds appear before first rounds for some competitions. To fix this, I import the <i>RoundTypes</i> table, containing an index - the <i>rank</i> - that will allow me to sort them correctly, and merge it with the dataset:

In [8]:
rounds = pd.read_csv('WCA_export_RoundTypes.tsv', sep='\t', low_memory = False)

In [9]:
df = df.merge(rounds[['id','rank']], how='left', left_on='roundTypeId', right_on='id')
df = df.drop('id',1)

The final dataset looks like this:

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2870882 entries, 0 to 2870881
Data columns (total 38 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   competitionId          object
 1   eventId                object
 2   roundTypeId            object
 3   pos                    int64 
 4   best                   int64 
 5   average                int64 
 6   personName             object
 7   personId               object
 8   personCountryId        object
 9   formatId               object
 10  value1                 int64 
 11  value2                 int64 
 12  value3                 int64 
 13  value4                 int64 
 14  value5                 int64 
 15  regionalSingleRecord   object
 16  regionalAverageRecord  object
 17  competitionName        object
 18  cityName               object
 19  countryId              object
 20  information            object
 21  year                   int64 
 22  month                  int64 
 23  day    

### List of competitions

In [11]:
comps = df[df['personId'] == wcaid].copy(deep = True)

#add dates
comps['date'] = pd.to_datetime(comps[['year','month','day']])

#competitors
comps['Competitors'] = df.groupby('competitionId')['personId'].transform('nunique')

#filter comps
comps = comps.drop_duplicates(subset = 'competitionId', keep = 'first')

#convert latitude to decimal
comps['latitude'] = comps['latitude'] / 1000000
comps['longitude'] = comps['longitude'] / 1000000

In [12]:
#pick data
comps = comps[['personId','personName','competitionId','date','latitude','longitude','Competitors','countryId']].sort_values(by = 'date', ascending = True)

#Remove multi-venue competitions
comps = comps[~(comps['countryId'].isin(['XA','XE','XF','XM','XN','XO','XS','XW']))].reset_index(drop = True)

#cleaning
comps = comps.rename(columns = {'personId':'WCAID','personName':'Name','competitionId':'Competition','countryId':'Country'})
comps.index += 1

In [13]:
#label competitions based on the number of competitors
comps['competitors_cat'] = ['A' if x<=50 else 'B' if 50<x<=100 else 'C' if 100<x<=150 else 'D' if 150<x<=200 else 'E' for x in comps['Competitors']]

In [14]:
#label delegated competitions
delegated = df[df['wcaDelegate'].str.contains(comps['Name'].iloc[0])].drop_duplicates(subset = 'competitionId', keep = 'first')
delegated = list(set(delegated['competitionId']))

comps['delegated'] = ['Y' if x in delegated else 'N' for x in comps['Competition']]

In [15]:
#label competitions with podiums
podiums = df[(df['personId'] == wcaid) & (df['pos']<=3) & (df['best']>0) & (df['roundTypeId'].isin(['c','f']))].drop_duplicates(subset = 'competitionId', keep = 'first')
podiums = list(set(podiums['competitionId']))

comps['podium'] = ['Y' if x in podiums else 'N' for x in comps['Competition']]

In [16]:
#distance from previous competition
from math import sin, cos, sqrt, atan2, radians

def distance(lat1,lon1,lat2,lon2):
    
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c



dist_l = [0]
for k in range(1,comps.shape[0]):
    d = distance(comps.iloc[k-1, 4], comps.iloc[k-1, 5], comps.iloc[k, 4], comps.iloc[k, 5])
    dist_l.append(d)
    
comps['distance'] = dist_l
comps['distance_cat'] = ['A' if x<=10 else 'B' if 10<x<=200 else 'C' if 200<x<=500 else 'D' if 500<x<=1000 else 'E' for x in comps['distance']]


In [17]:
#add records label
#map
records_dict = {'NR':1, 'AfR':2, 'NAR':2, 'AsR':2, 'OcR':2, 'ER':2, 'SAR':2, 'WR':3}

df['record_s'] = df['regionalSingleRecord'].map(records_dict)
df['record_a'] = df['regionalAverageRecord'].map(records_dict)

#filter
records = df[(df['personId'] == wcaid) & ((df['record_s']>0) | (df['record_a']))].copy(deep = True)
records = records.fillna(0)
records['record_code'] = [int(x) if x>y else int(y) for (x,y) in zip(records['record_s'],records['record_a'])]

#clean
records = records[['competitionId','record_code','eventId']]
records = records.sort_values(by = ['competitionId', 'record_code'], ascending = [True,False])
records = records.drop_duplicates(subset = 'competitionId', keep = 'first')
records = records.rename(columns = {'eventId':'event_record'})

#merge
comps = comps.merge(records, how = 'left', left_on = 'Competition', right_on = 'competitionId')
comps = comps.drop('competitionId', 1)
comps = comps.fillna(0)
comps['record_code'] = comps['record_code'].astype(int)

In [18]:
comps

,WCAID,Name,Competition,date,latitude,longitude,Competitors,Country,competitors_cat,delegated,podium,distance,distance_cat,record_code,event_record
0,2003BRUC01,Ron van Bruchem,WC2003,2003-08-23,43.716470,-79.338711,88,Canada,B,Y,Y,0.000000,A,2,minx
1,2003BRUC01,Ron van Bruchem,DutchOpen2003,2003-10-11,51.405966,5.414813,7,Netherlands,A,Y,Y,6056.779205,E,0,0
2,2003BRUC01,Ron van Bruchem,GermanOpen2004,2004-04-24,51.909060,8.375980,13,Germany,A,Y,Y,211.835319,C,2,minx
3,2003BRUC01,Ron van Bruchem,Euro2004,2004-08-07,52.374238,4.911994,53,Netherlands,B,Y,Y,242.031196,C,1,333
4,2003BRUC01,Ron van Bruchem,DutchOpen2004,2004-10-10,53.214882,6.567757,17,Netherlands,A,Y,Y,145.406114,B,1,333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,2003BRUC01,Ron van Bruchem,DutchMasters2020,2020-02-01,52.294335,5.637006,159,Netherlands,D,Y,N,1760.580927,E,0,0
183,2003BRUC01,Ron van Bruchem,StarCubingNijmegen2020,2020-02-29,51.871055,5.868428,140,Netherlands,C,Y,N,49.667613,B,0,0
184,2003BRUC01,Ron van Bruchem,AUAOpen2021,2021-12-10,40.193339,44.504413,52,Armenia,B,Y,N,3206.506976,E,0,0
185,2003BRUC01,Ron van Bruchem,DutchSpring2022,2022-04-17,52.028423,5.532623,184,Netherlands,D,Y,N,3231.847438,E,0,0


### Creating the Map

Bokeh works with the Web Mercator Format. This function converts latitude and longitude.

In [19]:
def wgs84_to_web_mercator(df, lon="longitude", lat="latitude"):
    """Converts decimal longitude/latitude to Web Mercator format"""
    k = 6378137
    df["mercator_x"] = df[lon] * (k * np.pi/180.0)
    df["mercator_y"] = np.log(np.tan((90 + df[lat]) * np.pi/360.0)) * k
    return df

In [20]:
comps = wgs84_to_web_mercator(comps) #conversion

I create an empty plot that includes all the competitions

In [21]:
# web mercator coordinates
shift = 100000
x_range,y_range = ((comps["mercator_x"].min() - shift, comps["mercator_x"].max() + shift), 
                  (comps["mercator_y"].min() - shift, comps["mercator_y"].max() + shift))

This allows to request for specific tiles once the plot is created

In [22]:
url = 'http://a.basemaps.cartocdn.com/rastertiles/voyager/{Z}/{X}/{Y}.png'
attribution = "Tiles by Carto, under CC BY 3.0. Data by OSM, under ODbL"

### Filling the map

### Plot

In [23]:
output_file(filename = f"CompMap{wcaid}.html")

In [24]:
fonte = ColumnDataSource(comps.to_dict(orient='list'))

#Hovertool
hover = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition"),("Competitors", "@Competitors")])

#filters
viewA = CDSView(source=fonte, filters=[GroupFilter(column_name='competitors_cat', group='A')])
viewB = CDSView(source=fonte, filters=[GroupFilter(column_name='competitors_cat', group='B')])
viewC = CDSView(source=fonte, filters=[GroupFilter(column_name='competitors_cat', group='C')])
viewD = CDSView(source=fonte, filters=[GroupFilter(column_name='competitors_cat', group='D')])
viewE = CDSView(source=fonte, filters=[GroupFilter(column_name='competitors_cat', group='E')])

#create figure
p = figure(tools=['pan, wheel_zoom, box_zoom, reset',hover], title ='Interactive map of '+str(wcaid)+"'s competitions", 
            x_range=x_range, y_range=y_range, x_axis_type="mercator", y_axis_type="mercator", width=800)

#add tiles
p.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add legend
p.add_layout(Legend(), 'right')
p.legend.click_policy="mute"
p.legend.title = 'Number of Competitors'
p.legend.title_text_font_style = 'bold'


#add glyphs
p.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewA, fill_color = Spectral5[0], muted_alpha=0.2, line_color = 'black', legend_label='≤ 50')
p.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewB, fill_color = Spectral5[1], muted_alpha=0.2, line_color = 'black', legend_label='≤ 100')
p.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewC, fill_color = Spectral5[2], muted_alpha=0.2, line_color = 'black', legend_label='≤ 150')
p.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewD, fill_color = Spectral5[3], muted_alpha=0.2, line_color = 'black', legend_label='≤ 200')
p.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewE, fill_color = Spectral5[4], muted_alpha=0.2, line_color = 'black', legend_label='> 200')

#tab1
tab1 = Panel(child=p, title="Number of Competitors")


In [25]:
#Hovertool
hover2 = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition"),("Delegated", "@delegated")])

#filters
viewY = CDSView(source=fonte, filters=[GroupFilter(column_name='delegated', group='Y')])
viewN = CDSView(source=fonte, filters=[GroupFilter(column_name='delegated', group='N')])

#create figure
p2 = figure(tools=['pan, wheel_zoom, box_zoom, reset',hover2], title ='Interactive map of '+str(wcaid)+"'s competitions", 
            x_range=x_range, y_range=y_range, x_axis_type="mercator", y_axis_type="mercator", width=800)

#add tiles
p2.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add legend
p2.add_layout(Legend(), 'right')
p2.legend.click_policy="hide"
p2.legend.title = 'Delegated Competitions'
p2.legend.title_text_font_style = 'bold'


#add glyphs
p2.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewY, fill_color = 'yellow', muted_alpha=0.2, line_color = 'black', legend_label='Delegated')
p2.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewN, fill_color = 'green', alpha = 0.8, muted_alpha=0.2, line_color = 'black', legend_label='Not Delegated')

#tab2
tab2 = Panel(child=p2, title="Delegated Competitions")


In [26]:
#Hovertool
hover3 = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition"),("Podium", "@podium")])

#filters
viewYp = CDSView(source=fonte, filters=[GroupFilter(column_name='podium', group='Y')])
viewNp = CDSView(source=fonte, filters=[GroupFilter(column_name='podium', group='N')])

#create figure
p3 = figure(tools=['pan, wheel_zoom, box_zoom, reset',hover3], title ='Interactive map of '+str(wcaid)+"'s competitions", 
            x_range=x_range, y_range=y_range, x_axis_type="mercator", y_axis_type="mercator", width=800)

#add tiles
p3.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add legend
p3.add_layout(Legend(), 'right')
p3.legend.click_policy="hide"
p3.legend.title = 'Podiums'
p3.legend.title_text_font_style = 'bold'


#add glyphs
p3.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewYp, fill_color = '#FCC201', line_color = 'black', legend_label='Podium')
p3.circle(source = fonte, x='mercator_x', y='mercator_y', size=10, view = viewNp, fill_color = 'white', line_color = 'black', legend_label='No Podium')

#tab2
tab3 = Panel(child=p3, title="Podiums")

In [27]:
#colours (multiple line calls don't work with views)
dict_colour = {"A":Spectral5[0], "B":Spectral5[1], "C":Spectral5[2], "D":Spectral5[3], "E":Spectral5[4]}
comps["colour"] = comps["distance_cat"].map(dict_colour)

#data for lines
comps["mercator_x_s"] = comps["mercator_x"].shift(1)
comps["mercator_y_s"] = comps["mercator_y"].shift(1)
comps["x"] = comps[["mercator_x", "mercator_x_s"]].values.tolist()
comps["y"] = comps[["mercator_y", "mercator_y_s"]].values.tolist()

fonte = ColumnDataSource(comps.to_dict(orient='list'))

#create figure
p4 = figure(tools=['pan, wheel_zoom, box_zoom, reset'], title ='Interactive map of '+str(wcaid)+"'s competitions", 
            x_range=x_range, y_range=y_range, x_axis_type="mercator", y_axis_type="mercator", width=800)

#add tiles
p4.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add glyphs
p4.circle(x='mercator_x', y='mercator_y', source = fonte, size=10, fill_color = 'blue', line_color = 'black')
p4.multi_line('x', 'y', source = fonte, line_width=2, line_color = 'colour')

#tab4
tab4 = Panel(child=p4, title="Graph")

In [28]:
dict_records_rev = {0:'no', 1:'NR', 2:'CR', 3:'WR'}
comps['record_cat'] = comps["record_code"].map(dict_records_rev)

fonte5 = ColumnDataSource(comps.to_dict(orient='list'))

#Hovertool
hover5 = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition"),("Event", "@event_record")])

#filters
viewA_r = CDSView(source=fonte5, filters=[GroupFilter(column_name='record_cat', group='no')])
viewB_r = CDSView(source=fonte5, filters=[GroupFilter(column_name='record_cat', group='NR')])
viewC_r = CDSView(source=fonte5, filters=[GroupFilter(column_name='record_cat', group='CR')])
viewD_r = CDSView(source=fonte5, filters=[GroupFilter(column_name='record_cat', group='WR')])

#create figure
p5 = figure(tools=['pan, wheel_zoom, box_zoom, reset',hover5], title ='Interactive map of '+str(wcaid)+"'s competitions", 
            x_range=x_range, y_range=y_range, x_axis_type="mercator", y_axis_type="mercator", width=800)

#add tiles
p5.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add legend
p5.add_layout(Legend(), 'right')
p5.legend.click_policy="hide"
p5.legend.title = 'Records'
p5.legend.title_text_font_style = 'bold'


#add glyphs
p5.circle(source = fonte5, x='mercator_x', y='mercator_y', size=10, view = viewA_r, fill_color = 'white', line_color = 'black', legend_label='No record')
p5.circle(source = fonte5, x='mercator_x', y='mercator_y', size=10, view = viewB_r, fill_color = '#28A745', line_color = 'black', legend_label='NR')
p5.circle(source = fonte5, x='mercator_x', y='mercator_y', size=10, view = viewC_r, fill_color = '#DC474C', line_color = 'black', legend_label='CR')
p5.circle(source = fonte5, x='mercator_x', y='mercator_y', size=10, view = viewD_r, fill_color = '#0366D6', line_color = 'black', legend_label='WR')


#tab5
tab5 = Panel(child=p5, title="Records")


In [29]:
source = ColumnDataSource(comps.to_dict(orient='list'))

columns = [
        TableColumn(field="date", title="Date", formatter=DateFormatter()),
        TableColumn(field="Competition", title="Competition"),
        TableColumn(field="Country", title="Country"),
        TableColumn(field="Competitors", title="Competitors"),
]

data_table = DataTable(source=source, columns=columns, sortable = True, width=800, height=400)





#tab6
tab6 = Panel(child=data_table, title="Competitions")

In [30]:
div = Div(text="""

<center><h1> Info </h1></center>

These plots are set up so that for every WCAID, all the competitions are included. Unfortunately, if you've been all around the world, you wll have to scroll and zoom to make it readable.
<br><br>
In each of the tabs, you can hover over data points to see additional information such as the name of the competition or the event in which you had a record.<br>
Legend are clickable: by clicking on a specific entry, you will hide the corresponding data points.
<br><br>
The graph tab shows competitions linked one after the other. The darker the color, the farther apart those competitions were.
<br><br>
Green button takes you to this person's WCA profile.<br>
Blue button takes you to my github in which you will find the notebook to make your own map.<br>
If you have Anaconda installed, all the libraries should already be there.
<br><br>
Have fun!
<br><br>
Tommaso, 2014RAPO01
""",
width=800, height=400)

#tab7
tab7 = Panel(child = div, title = 'Info')

In [31]:
#button
b1 = Button(label="WCA Profile", button_type="success", width = 400, align = 'center')
b1.js_on_click(
    CustomJS(
        args = dict(wcaid = wcaid),
        code = "window.open(`https://www.worldcubeassociation.org/persons/"+str(wcaid)+"`,\"_blank\");"
    )
)
b2 = Button(label="Generate your own map", button_type="primary", width = 400, align = 'center')
b2.js_on_click(
    CustomJS(
        args = dict(wcaid = wcaid),
        code = "window.open(`https://github.com/TRaposio/wca-statistics`,\"_blank\");"
    )
)

save(column(b1, b2, Tabs(tabs=[tab1, tab2, tab3, tab5, tab4, tab6, tab7])))

reset_output()